# Import data & library

In [1]:
YES = 1
NO = 0

DATA_DIR = '../' # Change to your DATA PATH

In [2]:
import tensorflow as tf
import sys

print(f"Python Version: {sys.version}")
print(f"TensorFlow Version: {tf.__version__}")
gpu_devices = tf.config.list_physical_devices('GPU')
print(f"Available GPUs: {gpu_devices}")

%matplotlib inline

2025-11-04 00:37:48.583734: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-04 00:37:48.746036: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-04 00:37:49.042844: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-11-04 00:37:49.042884: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-11-04 00:37:49.084645: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to

Python Version: 3.10.18 (main, Jun  5 2025, 13:14:17) [GCC 11.2.0]
TensorFlow Version: 2.15.1
Available GPUs: []


In [3]:
import os
import numpy as np
import matplotlib.pyplot as plt
import h5py
from tqdm import tqdm
import tensorflow as tf
from qkeras import *

import keras_tuner as kt
from qkeras import QConv2DBatchnorm, QActivation, QDense
import json


import random

def set_seed(seed=42):
    tf.random.set_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

In [4]:
RHEED_data_file = 'RHEED_4848_test6.h5' # DATA_DIR + 'RHEED_4848_test6.h5'
spot = 'spot_2'

h5 = h5py.File(RHEED_data_file, 'r')

raw_data = []
for growth in h5.keys():
    raw_data.extend(h5[growth][spot])
raw_data = np.array(raw_data).astype(np.float32)
raw_data = np.expand_dims(raw_data, axis=-1).astype(np.float32) # if (batch_size, height, width, channels)

print(f'[Raw Images Shape]: {raw_data.shape}')

[Raw Images Shape]: (150985, 48, 48, 1)


In [5]:
# Normalize w/ image max

normalized_images = []
normalized_factor = []
for image in tqdm(raw_data):
    normalized_images.append(image / (np.max(image) + 1))
    normalized_factor.append(np.max(image))
normalized_images = np.array(normalized_images).astype(np.float32)
normalized_factor = np.array(normalized_factor).astype(np.float32)

print(f'[Normalized Images Shape]: {normalized_images.shape}')

100%|██████████| 150985/150985 [00:01<00:00, 94978.44it/s]


[Normalized Images Shape]: (150985, 48, 48, 1)


In [6]:
# Clamp to 8-bit fractional
normalized_images_0I_8F = np.round(normalized_images * 256) / 256
normalized_images_0I_8F = normalized_images_0I_8F.astype(np.float32)

estimated_label = np.load("../Estimated_Labels.npy").astype(np.float32)

# shuffle the data
idx = np.random.permutation(len(normalized_images_0I_8F))
shuffled_data = normalized_images_0I_8F[idx]
shuffled_label = estimated_label[idx]

# Split into train/val
split_num = int(0.9 * len(idx))
train_data = shuffled_data[:split_num]
train_label = shuffled_label[:split_num]
val_data = shuffled_data[split_num:]
val_label = shuffled_label[split_num:]


print(f"Training Data Shape   : {train_data.shape}")
print(f"Validation Data Shape : {val_data.shape}")

# === Create tf.data datasets ===
BATCH_SIZE = 512
train_dataset = tf.data.Dataset.from_tensor_slices((train_data, train_label)).shuffle(len(train_data)).batch(BATCH_SIZE)
val_dataset = tf.data.Dataset.from_tensor_slices((val_data, val_label)).batch(BATCH_SIZE)

# === Training Config ===
train_model = YES
save_model = YES
load_model = NO
# Gaussian_Model_QAT_file = DATA_DIR + '/new_models/model_3conv_little.keras'


FileNotFoundError: [Errno 2] No such file or directory: '../Estimated_Labels.npy'

In [ ]:
# Gaussian Functions: (TENSORFLOW)
print_example_guassian = NO
print_example_loss = NO

# mean_x, mean_y, cov_x, cov_y, theta
def generate_guassian(batch, image_shape):
    batch_size = batch.shape[0]
    batch = tf.expand_dims(tf.expand_dims(batch, axis=-1), axis=-1)
    x0, y0, sigma_x, sigma_y, theta = tf.cast(tf.unstack(batch, axis=-3), tf.float32)
    
    x_range = tf.range(start=0, limit=image_shape[0], delta=1)
    y_range = tf.range(start=0, limit=image_shape[1], delta=1)
    X_coord, Y_coord = tf.meshgrid(x_range, y_range, indexing='xy')
    X_coord = tf.cast(tf.expand_dims(X_coord, axis=0), tf.float32)
    Y_coord = tf.cast(tf.expand_dims(Y_coord, axis=0), tf.float32)
    
    X_coord = tf.tile(X_coord, [batch_size, 1, 1])
    Y_coord = tf.tile(Y_coord, [batch_size, 1, 1])
    
    a = tf.math.pow(tf.math.cos(theta), 2) / (2 * tf.math.pow(sigma_x, 2)) + tf.math.pow(tf.math.sin(theta), 2) / (2 * tf.math.pow(sigma_y, 2))
    b = -1 * tf.math.sin(theta) * tf.math.cos(theta) / (2 * tf.math.pow(sigma_x, 2)) + tf.math.sin(theta) * tf.math.cos(theta) / (2 * tf.math.pow(sigma_y, 2))
    c = tf.math.pow(tf.math.sin(theta), 2) / (2 * tf.math.pow(sigma_x, 2)) + tf.math.pow(tf.math.cos(theta), 2) / (2 * tf.math.pow(sigma_y, 2))

    img = tf.exp(-1 * (a * (X_coord - x0) ** 2 + 2 * b * (X_coord - x0) * (Y_coord - y0) + c * (Y_coord - y0) ** 2))

    return tf.expand_dims(img, axis=-1) # if (batch_size, height, width, channels)
    return tf.expand_dims(img, axis=1)  # if (batch_size, channels, height, width)

def custom_weighted_mse_loss(I, J, n):
  W = tf.pow(I, n)

  squared_diffs = tf.pow(I - J, 2)

  weighted_squared_diffs = W * squared_diffs

  loss = tf.reduce_mean(weighted_squared_diffs)

  return loss

if print_example_guassian:
    image_shape = (48, 48)
    batch = tf.convert_to_tensor([
        [24, 24, 13, 7, 0],
        [24, 20, 13, 7, 0],
        [24, 24, 13, 7, np.pi/2],
    ])

    generated_imgs = generate_guassian(batch, image_shape)
    for i in range(len(batch)):
        plt.subplot(1, len(batch), i + 1)
        plt.imshow(tf.squeeze(generated_imgs[i]))
    plt.show()

if print_example_loss:
    I = tf.random.normal((5, 1, 48, 48))
    J = tf.random.normal((5, 1, 48, 48))
    n = 2
    loss = custom_weighted_mse_loss(I, J, n)
    print("[Custom Weighted MSE Loss]:", loss.numpy())
    

In [ ]:
# def dice_loss(y_true, y_pred, delta=0.6):   #300epoch 82% delta = 1
#     error = y_true - y_pred
#     is_small = tf.abs(error) <= delta
#     squared_loss = 0.5 * tf.square(error)
#     linear_loss = delta * (tf.abs(error) - 0.5 * delta)
#     return tf.reduce_mean(tf.where(is_small, squared_loss, linear_loss))


scaling_arr = np.array([48.0, 48.0, 24.0, 24.0, np.pi/2], dtype=np.float32)

# huber loss
def dice_loss(y_true, y_pred, delta=0.5):

    weights = tf.constant([3.0, 1.0, 2.0, 2.0, 0.1], dtype=tf.float32)
    y_pred = y_pred * scaling_arr + 1e-8
    diff = y_true - y_pred
    abs_diff = tf.abs(diff)

    quadratic = tf.minimum(abs_diff, delta)
    linear = abs_diff - quadratic
    huber_loss = 0.5 * tf.square(quadratic) + delta * linear

    weighted_loss = weights * huber_loss
    return tf.reduce_mean(weighted_loss)

# Import model & check performance

In [ ]:
# custom_objects = {
#     'dice_loss': dice_loss,
#     'QConv2DBatchnorm': QConv2DBatchnorm,
#     'QActivation': QActivation,
#     'QDense': QDense,
# }

# keras_path   = DATA_DIR + '/new_models/model_good1.keras'
# weights_path = DATA_DIR + '/new_models/model_good1_weights.h5'

# with tf.keras.utils.custom_object_scope(custom_objects):
#     ref_model = tf.keras.models.load_model(keras_path, compile=False)

# ref_model.save_weights(weights_path)        
# print("✓  已把純權重存成", weights_path)

In [ ]:
# def show_quant_bits(m):
#     for layer in m.layers:
#         qk = getattr(layer, "kernel_quantizer", None)
#         qb = getattr(layer, "bias_quantizer",   None)
#         qa = getattr(layer, "activation",       None)  
#         if qk or qb or isinstance(layer, QActivation):
#             print(f"{layer.name:25s}",
#                   f"kq={getattr(qk, 'get_config', lambda: qk)()}",
#                   f"bq={getattr(qb, 'get_config', lambda: qb)()}",
#                   f"aq={getattr(qa, 'get_config', lambda: qa)()}"
#                   if isinstance(layer, QActivation) else "")
            
# show_quant_bits(ref_model)

In [ ]:
import tensorflow as tf

TOTAL_BITS        = 8
INTEGER_FIRST     = 0   
INTEGER_REST      = 2  
L1_REG            = 1e-4

def build_model(input_shape=(48, 48, 1)):

    model = tf.keras.Sequential(
        [

            QConv2DBatchnorm(
                input_shape=input_shape,
                filters=6, kernel_size=3, strides=1, padding="valid",
                kernel_quantizer=f"quantized_bits({TOTAL_BITS},{0},alpha=1)",
                bias_quantizer=f"quantized_bits({TOTAL_BITS},{0},alpha=1)",
                kernel_initializer="lecun_uniform",
                kernel_regularizer=tf.keras.regularizers.l1(L1_REG),
                use_bias=True,
            ),
            QActivation(f"quantized_relu({TOTAL_BITS}, {INTEGER_REST})"),
            tf.keras.layers.MaxPool2D(pool_size=4, strides=4),

            QConv2DBatchnorm(
                filters=8, kernel_size=3, strides=1, padding="valid",
                kernel_quantizer=f"quantized_bits({TOTAL_BITS},{INTEGER_REST},alpha=1)",
                bias_quantizer=f"quantized_bits({TOTAL_BITS},{INTEGER_REST},alpha=1)",
                kernel_initializer="lecun_uniform",
                kernel_regularizer=tf.keras.regularizers.l1(L1_REG),
                use_bias=True,
            ),
            QActivation(f"quantized_relu({TOTAL_BITS}, {INTEGER_REST})"),
            tf.keras.layers.MaxPool2D(pool_size=2, strides=2),

            QConv2DBatchnorm(
                filters=10, kernel_size=3, strides=1, padding="valid",
                kernel_quantizer=f"quantized_bits({TOTAL_BITS},{INTEGER_REST},alpha=1)",
                bias_quantizer=f"quantized_bits({TOTAL_BITS},{INTEGER_REST},alpha=1)",
                kernel_initializer="lecun_uniform",
                kernel_regularizer=tf.keras.regularizers.l1(L1_REG),
                use_bias=True,
            ),
            QActivation(f"quantized_relu({TOTAL_BITS}, {INTEGER_REST})"),
            tf.keras.layers.MaxPool2D(pool_size=2, strides=2),

            tf.keras.layers.Flatten(),
            QDense(units=15, 
                   kernel_quantizer=f"quantized_bits({TOTAL_BITS},{INTEGER_REST},alpha=1)", 
                   bias_quantizer=f"quantized_bits({TOTAL_BITS},{INTEGER_REST},alpha=1)"),
            QActivation(f"quantized_relu({TOTAL_BITS}, {INTEGER_REST})"),
            QDense(units=10, 
                   kernel_quantizer=f"quantized_bits({TOTAL_BITS},{INTEGER_REST},alpha=1)", 
                   bias_quantizer=f"quantized_bits({TOTAL_BITS},{INTEGER_REST},alpha=1)"),
            QActivation(f"quantized_relu({TOTAL_BITS}, {INTEGER_REST})"),
            QDense(units=5 , 
                   kernel_quantizer=f"quantized_bits({TOTAL_BITS},{INTEGER_REST},alpha=1)", 
                   bias_quantizer=f"quantized_bits({TOTAL_BITS},{INTEGER_REST},alpha=1)"),
        ],
        name="model_good1_seq"
    )
    
    return model


model = build_model()      

# change the weight path here
weights_path = DATA_DIR + "/new_models/model_good1.weights.h5"

model.load_weights(weights_path)
model.compile(optimizer="adam", loss=dice_loss, run_eagerly=True)
model.summary()

Model: "model_good1_seq"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 q_conv2d_batchnorm (QConv2D  (None, 46, 46, 6)        85        
 Batchnorm)                                                      
                                                                 
 q_activation (QActivation)  (None, 46, 46, 6)         0         
                                                                 
 max_pooling2d (MaxPooling2D  (None, 11, 11, 6)        0         
 )                                                               
                                                                 
 q_conv2d_batchnorm_1 (QConv  (None, 9, 9, 8)          473       
 2DBatchnorm)                                                    
                                                                 
 q_activation_1 (QActivation  (None, 9, 9, 8)          0         
 )                                                 

In [ ]:
# custom_objects = {
#     'dice_loss': dice_loss,
#     'QConv2DBatchnorm': QConv2DBatchnorm,
#     'QActivation': QActivation,
#     'QDense': QDense,
# }

# weights_path = DATA_DIR + "/new_models/model_good1.weights.h5"

# # with tf.keras.utils.custom_object_scope(custom_objects):
# model = build_model()      
# model.load_weights(weights_path)

In [ ]:
# model.summary()

In [ ]:
# # Model_file = DATA_DIR + '/new_models/Gaussian_Model_QAT_hp5.keras'
# # Model_file = DATA_DIR + '/new_models/model_manual.keras'
# Model_file = DATA_DIR + '/new_models/model_good1.keras'
# # Model_file = DATA_DIR + '/Models/model.keras'
# # bad4 22000 dice_loss 96%
# # model 7300 dice_loss 96%


# # with tf.keras.utils.custom_object_scope({'custom_weighted_mse_loss': custom_weighted_mse_loss,
# #                                          'QConv2DBatchnorm': QConv2DBatchnorm,
# #                                          'QActivation': QActivation,
# #                                          'QDense': QDense
# #                                          }):
# #         model = tf.keras.models.load_model(Model_file)


# with tf.keras.utils.custom_object_scope({'dice_loss': dice_loss,
#                                          'QConv2DBatchnorm': QConv2DBatchnorm,
#                                          'QActivation': QActivation,
#                                          'QDense': QDense
#                                          }):
#         model = tf.keras.models.load_model(Model_file)

# model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 q_conv2d_batchnorm (QConv2D  (None, 46, 46, 6)        85        
 Batchnorm)                                                      
                                                                 
 q_activation (QActivation)  (None, 46, 46, 6)         0         
                                                                 
 max_pooling2d (MaxPooling2D  (None, 11, 11, 6)        0         
 )                                                               
                                                                 
 q_conv2d_batchnorm_1 (QConv  (None, 9, 9, 8)          473       
 2DBatchnorm)                                                    
                                                                 
 q_activation_1 (QActivation  (None, 9, 9, 8)          0         
 )                                                      

In [ ]:
def evaluate_prediction(data, label, name=""):
    predicted = model.predict(data, batch_size=256) * scaling_arr
    label = label.copy()

    predicted[:, 2:4] = np.abs(predicted[:, 2:4])
    label[:, 2:4] = np.abs(label[:, 2:4])

    diff = predicted - label
    abs_diff = np.abs(diff)

    # Identify outliers where any value exceeds threshold
    threshold = 100.0
    valid_mask = np.all(np.abs(label) < threshold, axis=1)
    invalid_mask = ~valid_mask

    # Report outliers
    print(f"\n===== {name} Data Evaluation =====")
    print(f"Total samples: {len(label)}")
    print(f"Valid samples: {np.sum(valid_mask)}")
    print(f"Outliers detected: {np.sum(invalid_mask)}")

    if np.any(invalid_mask):
        print("\n  Outlier label(s) (abs(value) >= 100):")
        for idx in np.where(invalid_mask)[0]:
            print(f"[{idx:5d}] → {label[idx]}")

    # Compute metrics on valid data only
    filtered_abs_diff = abs_diff[valid_mask]
    filtered_label = label[valid_mask]

    mae_diff = np.mean(filtered_abs_diff, axis=0)
    within_thresh = np.all(filtered_abs_diff[:, :4] < 1.0, axis=1)
    percent_within = 100.0 * np.sum(within_thresh) / len(filtered_abs_diff)

    param_names = ["x0", "y0", "sigma_x", "sigma_y", "theta"]
    print("\nMean Absolute Error (MAE):")
    for n, v in zip(param_names, mae_diff):
        print(f"{n:>8s}: {v:.4f}")

    print(f"\nSamples with first 4 params error < 1: {np.sum(within_thresh)} / {len(filtered_abs_diff)}")
    print(f"Percentage: {percent_within:.2f}%")





images_to_predict = 100000
scaling_arr = np.array([48.0, 48.0, 24.0, 24.0, np.pi/2], dtype=np.float32)

train_data_eval = train_data[:images_to_predict]
train_label_eval = shuffled_label[:split_num][:images_to_predict]
evaluate_prediction(train_data_eval, train_label_eval, name="Train")

val_data_eval = val_data[:images_to_predict]
val_label_eval = shuffled_label[split_num:][:images_to_predict]
evaluate_prediction(val_data_eval, val_label_eval, name="Val")

 29/391 [=>............................] - ETA: 0s  

2025-07-14 13:59:12.418509: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:424] Loaded cuDNN version 8907


391/391 [==============================] - 1s 2ms/step

===== Train Data Evaluation =====
Total samples: 100000
Valid samples: 100000
Outliers detected: 0

Mean Absolute Error (MAE):
      x0: 0.2248
      y0: 0.3545
 sigma_x: 0.3653
 sigma_y: 0.4201
   theta: 0.1105

Samples with first 4 params error < 1: 94188 / 100000
Percentage: 94.19%
59/59 [==============================] - 0s 2ms/step

===== Val Data Evaluation =====
Total samples: 15099
Valid samples: 15098
Outliers detected: 1

  Outlier label(s) (abs(value) >= 100):
[ 5406] → [-8.2714648e+02 -6.2999760e+01  9.7220288e+02  2.5123726e+01
 -9.4661601e-02]

Mean Absolute Error (MAE):
      x0: 0.2234
      y0: 0.3532
 sigma_x: 0.3631
 sigma_y: 0.4195
   theta: 0.1104

Samples with first 4 params error < 1: 14232 / 15098
Percentage: 94.26%
